In [ ]:
from distributed_processing.redis_connector import RedisConnector
from distributed_processing.worker import Worker

In [ ]:
REDIS_HOST = "localhost"
REDIS_PORT = 6379
REDIS_DB = 0
NAMESPACE = "tasks"

In [ ]:
redis_connector = RedisConnector(redis_host=REDIS_HOST, redis_port=REDIS_PORT, 
                                 redis_db=REDIS_DB, namespace=NAMESPACE)

# Clean the namespace in case previous runs left anything behind.
redis_connector.clean_namespace()

server = Worker(redis_connector)



In [ ]:
def add(x, y):
    return x + y

def mul(x, y):
    return x * y

def div(x, y):
    return x / y

func_dict1 = {"add" : add,
              "mul" : mul,
              "div" : div}

server.add_requests_queue("cola_1", func_dict1)


In [ ]:
def hola(nombre, calificativo="listo"):
    return f"Hola {nombre}, eres muy {calificativo}"

func_dict2 = {"hola": hola}

# The methods available in cola_2 are not published in the registry
server.add_requests_queue("cola_2", func_dict2, register=False)

In [ ]:
# Adds the queue "py_eval" with the method eval_py_function, which the
# client's rpc_async_fn / rpc_sync_fn use to run dill-serialized functions
# on the worker. SECURITY: it executes any Python code sent by clients;
# only on trusted infrastructure. See Worker.add_python_eval.
server.add_python_eval()

In [ ]:
# Publish every method of every queue in the registry 
server.update_methods_registry()

In [ ]:
# run() is the choice for developing and testing in a notebook: any connector
# error (Redis down, MISCONF, corrupt message...) stops the cell and shows the
# traceback. run(timeout=60) serves requests for 60 s and then returns.
#
# For a worker deployed as a service use run_forever(): the error is logged,
# the loop waits with exponential backoff and keeps listening. It stops with
# server.stop() (from another thread or from a registered function).
server.run()
# server.run_forever(backoff=(1, 60))